## Langkah 1: Cek & Verifikasi GPU
Jalankan cell ini untuk memastikan Colab Anda menggunakan GPU (T4 atau A100).

In [ ]:
import subprocess
res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if res.returncode == 0:
    print("=" * 60)
    print("🚀 GPU AKTIF! Siap untuk inferensi super cepat (< 1-2 detik)")
    print("=" * 60)
    print(res.stdout)
else:
    print("=" * 60)
    print("❌ PERINGATAN: GPU BELUM DIAKTIFKAN!")
    print("Colab Anda saat ini masih mode CPU biasa. Respon AI akan SANGAT LAMBAT (1-2 menit)!")
    print("=" * 60)
    print("👉 CARA AKTIFKAN GPU SEKARANG (HANYA 3 KLIK):")
    print("1. Klik menu 'Runtime' di atas (atau klik tombol 'Change runtime type' di kanan bawah)")
    print("2. Pada Hardware accelerator, pilih 'T4 GPU' (atau 'A100 GPU' jika punya Colab Pro)")
    print("3. Klik 'Save'")
    print("=" * 60)


## Langkah 2: Clone Repository

In [ ]:
import os
%cd /content
if os.path.exists('ShiroLLM-AI-CHAT'):
    print('📁 Folder ShiroLLM-AI-CHAT sudah ada. Mengambil update terbaru (git pull)...')
    %cd /content/ShiroLLM-AI-CHAT
    !git stash --quiet 2>/dev/null || true
    !git pull origin main
else:
    print('📥 Mengunduh repository ShiroLLM-AI-CHAT...')
    !git clone https://github.com/Renn-devBI/ShiroLLM-AI-CHAT.git
    %cd /content/ShiroLLM-AI-CHAT


## Langkah 3: Install Dependencies & Setup cuBLAS (Instan ⚡)
Menggunakan pre-compiled CUDA wheel resmi + perbaikan otomatis symlink cuBLAS (`libcublas.so.12`) sehingga **TIDAK AKAN ERROR LIBNOTFOUND**!

In [ ]:
# 1. Install dependencies web, pyngrok, & CUDA runtime + cuBLAS
!pip install -r requirements.txt

# 2. Install pre-compiled llama-cpp-python CUDA wheel (Hanya 5 detik!)
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 || pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

# 3. Konfigurasi Linker Linux & cuBLAS (Solusi Permanen Anti-Error libcublas.so.12!)
!python3 -c "import glob, os, subprocess; [os.symlink(lib, os.path.join('/usr/lib/x86_64-linux-gnu', os.path.basename(lib))) for lib in glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib/*.so*') + glob.glob('/usr/local/cuda*/lib64/*.so*') if not os.path.exists(os.path.join('/usr/lib/x86_64-linux-gnu', os.path.basename(lib)))]; subprocess.run(['ldconfig']); print('✓ Symlink cuBLAS & CUDA berhasil diperbarui!')"


## Langkah 4: Download 3 Model Utama Colab (Bisa Switch Kapan Saja di Web UI)
Agar Kakak bisa berganti model langsung di Web UI (menu **Pengaturan -> Model AI**):
- **Opsi 1 (Semua 3 Model)**: Set `MODEL_CHOICE = "all"` untuk mengunduh ketiga model sekaligus:
  1. **Qwen3-32B** (Model Utama ⭐: Penalaran tertinggi & paling cerdas, ~19.8 GB)
  2. **Qwen2.5-VL-7B** (Model Vision 👁️: Multimodal untuk kamera webcam VTuber & analisis gambar, ~5.6 GB)
  3. **Qwen2.5-7B** (Model Cepat ⚡: Ringan & super cepat di T4 GPU, ~5.4 GB)
- **Opsi 2**: Set `MODEL_CHOICE = "vl"` jika hanya ingin model Vision, atau `"32b"` untuk model utama saja.

In [ ]:
import os
# PILIHAN MODE DOWNLOAD:
# "all" -> Download ketiga model sekaligus (Bisa switch bebas langsung di Web UI!)
# "vl"  -> Download Model Vision Qwen2.5-VL-7B (Kamera webcam real-time & gambar VTuber)
# "32b" -> Download Model Utama Qwen3-32B (Penalaran tertinggi)
# "7b"  -> Download Model Cepat Qwen2.5-7B (Hemat VRAM T4)
os.environ["MODEL_CHOICE"] = "all"

!python download_model.py


## Langkah 5: 💾 Hubungkan Google Drive (Muat Ingatan & Data Training Shiro) ⭐
**SANGAT DIREKOMENDASIKAN**: Jalankan cell ini SEBELUM server dinyalakan!

### 🎯 Apa Saja yang Dimuat Otomatis dari Google Drive?
1. 🧠 **Ingatan Shiro (`ingatan_shiro.json` & `isekai_world.json`)**: Riwayat obrolan masa lalu, fakta yang diingat, status emosi & ikatan dengan Kakak.
2. 📚 **Data Pelatihan (`training_data/`)**: Seluruh dataset fine-tuning teks & multimodal vision yang pernah diekspor/dilatih sebelumnya akan langsung dibaca dan diintegrasikan ke pola respon aktif Shiro!
3. 🎯 **Model Adapters (`lora/`)**: Bobot LoRA hasil training (jika ada) akan otomatis terbaca dan diterapkan ke model.
4. 👤 **Profil & Avatar (`profile/`)**: Foto dan info profil Kakak & Shiro.
5. 🔑 **API Key Konsisten (`api_key.json`)**: Token API tidak akan berubah-ubah setiap kali restart.

⚡ **Fitur Real-Time Auto-Sync**: Setiap kali mengobrol di Web UI, ingatan dan data obrolan baru **langsung tersimpan otomatis** ke Google Drive Anda secara instan!

In [ ]:
from google.colab import drive
import os
import shutil
import json
import glob

# 1. Hubungkan Google Drive
print("🔄 Menghubungkan ke Google Drive...")
drive.mount('/content/drive')

# 2. Tentukan folder penyimpanan di Google Drive
DRIVE_DIR = '/content/drive/MyDrive/Shiro_Memory'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'profile'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'training_data'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'lora'), exist_ok=True)

# Beritahu server Shiro bahwa Google Drive aktif untuk auto-sync instan
os.environ["SHIRO_DRIVE_DIR"] = DRIVE_DIR

print("\n" + "=" * 62)
print("📂 Folder Penyimpanan Drive: Google Drive / Shiro_Memory")
print("=" * 62)

files_restored = []

# 3. Muat file ingatan lama dari Google Drive jika ada
drive_memory_file = os.path.join(DRIVE_DIR, 'ingatan_shiro.json')
if os.path.exists(drive_memory_file) and os.path.getsize(drive_memory_file) > 10:
    shutil.copy2(drive_memory_file, 'ingatan_shiro.json')
    files_restored.append('ingatan_shiro.json')
else:
    if os.path.exists('ingatan_shiro.json'):
        shutil.copy2('ingatan_shiro.json', os.path.join(DRIVE_DIR, 'ingatan_shiro.json'))

# 4. Muat isekai_world.json dari Google Drive jika ada
drive_world = os.path.join(DRIVE_DIR, 'isekai_world.json')
if os.path.exists(drive_world):
    shutil.copy2(drive_world, 'isekai_world.json')
    files_restored.append('isekai_world.json')
else:
    if os.path.exists('isekai_world.json'):
        shutil.copy2('isekai_world.json', os.path.join(DRIVE_DIR, 'isekai_world.json'))

# 5. Muat profil (user & AI avatar) dari Google Drive jika ada
drive_profile = os.path.join(DRIVE_DIR, 'profile')
if os.path.exists(drive_profile) and len(os.listdir(drive_profile)) > 0:
    shutil.copytree(drive_profile, 'profile', dirs_exist_ok=True)
    files_restored.append('profile/ (user & ai persona)')
else:
    if os.path.exists('profile'):
        shutil.copytree('profile', os.path.join(DRIVE_DIR, 'profile'), dirs_exist_ok=True)

# 6. Muat API key jika ada agar token konsisten
drive_api_key = os.path.join(DRIVE_DIR, 'api_key.json')
if os.path.exists(drive_api_key):
    shutil.copy2(drive_api_key, 'api_key.json')
    files_restored.append('api_key.json')

# 7. 🔥 MUAT DATASET TRAINING LAMA DARI GOOGLE DRIVE! 🔥
drive_training = os.path.join(DRIVE_DIR, 'training_data')
train_chat_count = 0
train_vision_count = 0

if os.path.exists(drive_training) and len(os.listdir(drive_training)) > 0:
    shutil.copytree(drive_training, 'training_data', dirs_exist_ok=True)
    files_restored.append('training_data/ (dataset pelatihan)')
else:
    if os.path.exists('training_data'):
        shutil.copytree('training_data', drive_training, dirs_exist_ok=True)

# Hitung jumlah sampel dataset training yang tersedia
for cand in ['training_data/shiro_alpaca_train.json', 'training_data/chat/shiro_alpaca_train.json']:
    if os.path.exists(cand):
        try:
            with open(cand, 'r', encoding='utf-8') as f:
                d = json.load(f)
                train_chat_count = max(train_chat_count, len(d) if isinstance(d, list) else 0)
        except Exception:
            pass

for cand in ['training_data/vision/shiro_vision_alpaca.json']:
    if os.path.exists(cand):
        try:
            with open(cand, 'r', encoding='utf-8') as f:
                d = json.load(f)
                train_vision_count = max(train_vision_count, len(d) if isinstance(d, list) else 0)
        except Exception:
            pass

# 8. 🚀 MUAT MODEL HASIL TRAINING LAMA (ShiroAI-LLM-V1, V2, dst.) DARI DRIVE
drive_model = os.path.join(DRIVE_DIR, 'model')
os.makedirs('model', exist_ok=True)
if os.path.exists(drive_model):
    for f in os.listdir(drive_model):
        if f.endswith('.gguf') and 'shiroai-llm-v' in f.lower():
            shutil.copy2(os.path.join(drive_model, f), os.path.join('model', f))
            files_restored.append(f'model/{f}')
        elif f.endswith('_info.json'):
            shutil.copy2(os.path.join(drive_model, f), os.path.join('model', f))

# 8. 🎯 MUAT BOBOT LORA ADAPTER DARI GOOGLE DRIVE (JIKA ADA)
drive_lora = os.path.join(DRIVE_DIR, 'lora')
lora_count = 0
if os.path.exists(drive_lora) and len(os.listdir(drive_lora)) > 0:
    shutil.copytree(drive_lora, 'lora', dirs_exist_ok=True)
    lora_files = glob.glob('lora/*.gguf') + glob.glob('lora/*.bin')
    lora_count = len(lora_files)
    if lora_count > 0:
        files_restored.append(f'lora/ ({lora_count} model adapter)')

# Baca info detail isi ingatan yang dimuat
try:
    with open('ingatan_shiro.json', 'r', encoding='utf-8') as f:
        mem_data = json.load(f)
        meta = mem_data.get('system_metadata', {})
        user_prof = mem_data.get('user_profile', {})
        user_name = user_prof.get('name', meta.get('user_name', 'Kakak'))
        turns_count = meta.get('total_turns', 0)
        facts_count = len(mem_data.get('knowledge_base', {}).get('facts', []))
        last_interaction = meta.get('last_interaction', 'Belum ada')
except Exception:
    user_name = "Kakak"
    turns_count = "?"
    facts_count = "?"
    last_interaction = "?"

print("🎉 INGATAN & DATA PELATIHAN BERHASIL DIMUAT DARI GOOGLE DRIVE!")
print(f"   👤 Panggilan Pengguna   : {user_name}")
print(f"   💬 Riwayat Percakapan   : {turns_count} pesan tersimpan")
print(f"   🧠 Fakta Pengetahuan    : {facts_count} fakta tersimpan")
print(f"   📚 Dataset Training Teks: {train_chat_count} percakapan latih")
print(f"   👁️ Dataset Vision/Gambar: {train_vision_count} multimodal latih")
if lora_count > 0:
    print(f"   🎯 Model LoRA Adapter   : {lora_count} adapter aktif")
print(f"   🕒 Interaksi Terakhir   : {last_interaction}")
print(f"   📦 Berkas Terpulihkan   : {', '.join(files_restored)}")

print("=" * 62)
print("⚡ REAL-TIME AUTO-SYNC AKTIF:")
print("   Setiap pesan baru saat chat otomatis tersimpan langsung ke Drive!")
print("👉 Silakan jalankan Langkah 6 untuk Menyalakan Server Shiro.")
print("=" * 62)


## Langkah 6 (Opsi A - NGROK): Jalankan Server dengan Ngrok ⭐
Gunakan opsi ini jika Anda memiliki token Ngrok (gratis di [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)).

In [ ]:
import os
import glob
import json
import subprocess
import time
import urllib.request
import getpass
import shutil
from pyngrok import ngrok
from IPython.display import clear_output

# 1. Pastikan libcudart.so.12 terdeteksi di LD_LIBRARY_PATH
cuda_libs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib') + \
            glob.glob('/usr/lib/python*/dist-packages/nvidia/*/lib') + \
            ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']
valid_paths = [p for p in cuda_libs if os.path.isdir(p)]
if valid_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(valid_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# 2. Muat atau buat API Key Shiro
from web import get_or_create_api_key
active_api_key = get_or_create_api_key()

# 3. Masukkan token Ngrok Anda di sini
NGROK_TOKEN = ""  # <-- Masukkan token Anda di dalam tanda kutip jika mau praktis
if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass("Masukkan Ngrok Authtoken Anda (dari dashboard.ngrok.com): ")

ngrok.set_auth_token(NGROK_TOKEN.strip())
ngrok.kill() # Tutup tunnel lama jika ada
public_url = ngrok.connect(7474).public_url

# 4. Offload model ke GPU Colab
os.environ["N_GPU_LAYERS"] = "-1"
os.environ["SHIRO_CONTEXT_SIZE"] = os.environ.get("SHIRO_CONTEXT_SIZE", "8192")

print("⏳ Sedang memuat model & menyalakan server Shiro ke GPU... Tunggu sebentar...")

# 5. Jalankan server di background & simpan log ke file shiro_server.log agar layar bersih
log_path = "/content/shiro_server.log" if os.path.exists("/content") else "shiro_server.log"
with open(log_path, "w", encoding="utf-8") as log_f:
    server_proc = subprocess.Popen(["python", "web.py"], stdout=log_f, stderr=log_f)

# 6. Tunggu sampai server siap merespons (maksimal 120 detik)
server_ready = False
for _ in range(120):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:7474/api/key", timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

# 7. AUTO CLEAR: Bersihkan semua log tensor/model berantakan dari layar Colab!
clear_output(wait=True)

if server_ready:
    print("\n" + "="*62)
    print("🎉 SERVER SHIRO AKTIF & SIAP DIGUNAKAN!")
    print("="*62)
    print(f"👉 BUKA WEB UI SHIRO DI SINI:")
    print(f"🔗 {public_url}")
    print("-"*62)
    print(f"🔑 KONEKSI BOT WHATSAPP / VTUBER / EKSTERNAL:")
    print(f"👉 SHIRO_BASE_URL = {public_url}/v1")
    print(f"👉 SHIRO_API_KEY  = {active_api_key}")
    print("="*62 + "\n")
    print("✨ Layar dibersihkan otomatis. Server sedang aktif berjalan...")
    print("📁 Catatan: Seluruh log sistem tersimpan rapi di 'shiro_server.log'.\n")
    try:
        server_proc.wait()
    except KeyboardInterrupt:
        server_proc.terminate()
        print("\nServer Shiro dihentikan.")
        # Sinkronisasi penutup ke Google Drive (ingatan, profil, data training)
        drive_dir = os.environ.get("SHIRO_DRIVE_DIR", "/content/drive/MyDrive/Shiro_Memory")
        if os.path.exists(drive_dir):
            for f in ['ingatan_shiro.json', 'isekai_world.json', 'api_key.json']:
                if os.path.exists(f):
                    shutil.copy2(f, os.path.join(drive_dir, f))
            if os.path.exists('profile'):
                shutil.copytree('profile', os.path.join(drive_dir, 'profile'), dirs_exist_ok=True)
            if os.path.exists('training_data'):
                shutil.copytree('training_data', os.path.join(drive_dir, 'training_data'), dirs_exist_ok=True)
            print("✓ Seluruh data, ingatan, & dataset training Shiro tersinkron aman ke Google Drive.")
else:
    print("❌ Gagal memulai server. Isi log terakhir:")
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            print(f.read()[-2000:])


## Langkah 6 (Opsi B - CLOUDFLARE): Jalankan Tanpa Perlu Akun / Token ⚡
Gunakan opsi ini jika ingin langsung membuka link tanpa perlu login / token Ngrok!

In [ ]:
import os
import glob
import subprocess
import time
import urllib.request
import re
import shutil
from IPython.display import clear_output

# 1. Pastikan libcudart.so.12 terdeteksi di LD_LIBRARY_PATH
cuda_libs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib') + \
            glob.glob('/usr/lib/python*/dist-packages/nvidia/*/lib') + \
            ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']
valid_paths = [p for p in cuda_libs if os.path.isdir(p)]
if valid_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(valid_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# 2. Muat atau buat API Key Shiro
from web import get_or_create_api_key
active_api_key = get_or_create_api_key()

# 3. Set GPU layers ke -1
os.environ["N_GPU_LAYERS"] = "-1"
os.environ["SHIRO_CONTEXT_SIZE"] = os.environ.get("SHIRO_CONTEXT_SIZE", "8192")

# 4. Download Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# 5. Jalankan tunnel Cloudflare
!rm -f /content/tunnel.log
tunnel_proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:7474"], 
                               stdout=open("/content/tunnel.log", "w"), 
                               stderr=subprocess.STDOUT)

print("⏳ Sedang memuat model & menyalakan server Shiro ke GPU... Tunggu sebentar...")

# 6. Jalankan server di background & simpan log ke file shiro_server.log agar layar bersih
log_path = "/content/shiro_server.log" if os.path.exists("/content") else "shiro_server.log"
with open(log_path, "w", encoding="utf-8") as log_f:
    server_proc = subprocess.Popen(["python", "web.py"], stdout=log_f, stderr=log_f)

# 7. Tunggu sampai link Cloudflare dan server lokal siap
cf_url = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
            if match:
                cf_url = match.group(1)
                break

server_ready = False
for _ in range(120):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:7474/api/key", timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

# 8. AUTO CLEAR: Bersihkan semua log berantakan dari layar Colab!
clear_output(wait=True)

if server_ready and cf_url:
    print("\n" + "="*62)
    print("🎉 SERVER SHIRO AKTIF & SIAP DIGUNAKAN!")
    print("="*62)
    print(f"👉 BUKA WEB UI SHIRO DI SINI:")
    print(f"🔗 {cf_url}")
    print("-"*62)
    print(f"🔑 KONEKSI BOT WHATSAPP / VTUBER / EKSTERNAL:")
    print(f"👉 SHIRO_BASE_URL = {cf_url}/v1")
    print(f"👉 SHIRO_API_KEY  = {active_api_key}")
    print("="*62 + "\n")
    print("✨ Layar dibersihkan otomatis. Server sedang aktif berjalan...")
    print("📁 Catatan: Seluruh log sistem tersimpan rapi di 'shiro_server.log'.\n")
    try:
        server_proc.wait()
    except KeyboardInterrupt:
        server_proc.terminate()
        tunnel_proc.terminate()
        print("\nServer Shiro dihentikan.")
        # Sinkronisasi penutup ke Google Drive (ingatan, profil, data training)
        drive_dir = os.environ.get("SHIRO_DRIVE_DIR", "/content/drive/MyDrive/Shiro_Memory")
        if os.path.exists(drive_dir):
            for f in ['ingatan_shiro.json', 'isekai_world.json', 'api_key.json']:
                if os.path.exists(f):
                    shutil.copy2(f, os.path.join(drive_dir, f))
            if os.path.exists('profile'):
                shutil.copytree('profile', os.path.join(drive_dir, 'profile'), dirs_exist_ok=True)
            if os.path.exists('training_data'):
                shutil.copytree('training_data', os.path.join(drive_dir, 'training_data'), dirs_exist_ok=True)
            print("✓ Seluruh data, ingatan, & dataset training Shiro tersinkron aman ke Google Drive.")
else:
    print(f"Link Cloudflare: {cf_url}")
    print("❌ Gagal memulai server. Isi log terakhir:")
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            print(f.read()[-2000:])


## ⚡ RERUN CEPAT / RESTART SERVER (Tanpa Hapus Sesi & Tanpa Download Ulang)
Jalankan cell di bawah ini kapan saja Kakak ingin:
1. **Mengambil update kode terbaru dari GitHub (`git pull`)**
2. **Merestart server Shiro yang sedang berjalan**

> [!TIP]
> **Aman 100% & Sangat Cepat (Hanya 3 - 5 Detik)**:
> - Seluruh riwayat obrolan, sesi chat, ingatan (`ingatan_shiro.json`), dan dataset training **TIDAK AKAN HILANG**.
> - Model AI yang sudah diunduh di folder `model/` **TETAP ADA** (tidak perlu download berjam-jam lagi!).
> - Dependensi Python (`pip`) **TIDAK PERLU DI-INSTALL ULANG**.


In [ ]:
# ==============================================================================
# ⚡ RERUN CEPAT & RESTART SERVER SHIRO (TANPA HAPUS SESI / DATA CHAT)
# ==============================================================================
import os
import subprocess
import time
import glob
import urllib.request
import re
import shutil
from IPython.display import clear_output

# PILIHAN TUNNEL: 'cloudflare' (gratis tanpa token) atau 'ngrok' (jika pakai token)
TUNNEL_MODE = 'cloudflare'

print('🛑 1. Menghentikan server lama & tunnel yang berjalan...')
!fuser -k 7474/tcp 2>/dev/null || true
!pkill -f 'python.*web.py' 2>/dev/null || true
!pkill -f 'cloudflared' 2>/dev/null || true
!pkill -f 'ngrok' 2>/dev/null || true
time.sleep(1)

%cd /content/ShiroLLM-AI-CHAT

print('💾 2. Mengamankan berkas ingatan, sesi, & dataset pelatihan...')
drive_dir = os.environ.get('SHIRO_DRIVE_DIR', '/content/drive/MyDrive/Shiro_Memory')
os.makedirs('/content/shiro_backup_sesi', exist_ok=True)

for item in ['ingatan_shiro.json', 'isekai_world.json', 'api_key.json']:
    if os.path.exists(item):
        shutil.copy2(item, f'/content/shiro_backup_sesi/{item}')
        if os.path.exists(drive_dir):
            shutil.copy2(item, os.path.join(drive_dir, item))

if os.path.exists('training_data'):
    shutil.copytree('training_data', '/content/shiro_backup_sesi/training_data', dirs_exist_ok=True)
    if os.path.exists(drive_dir):
        shutil.copytree('training_data', os.path.join(drive_dir, 'training_data'), dirs_exist_ok=True)

print('🔄 3. Mengambil pembaruan kode terbaru dari GitHub (git pull)...')
!git stash --quiet 2>/dev/null || true
!git pull origin main

# Pulihkan file sesi & ingatan agar tidak tertimpa template baru
for item in ['ingatan_shiro.json', 'isekai_world.json', 'api_key.json']:
    backup_path = f'/content/shiro_backup_sesi/{item}'
    if os.path.exists(backup_path):
        shutil.copy2(backup_path, item)

if os.path.exists('/content/shiro_backup_sesi/training_data'):
    shutil.copytree('/content/shiro_backup_sesi/training_data', 'training_data', dirs_exist_ok=True)

# Pulihkan model hasil pelatihan mandiri dari Google Drive jika ada
if drive_dir and os.path.exists(os.path.join(drive_dir, 'model')):
    os.makedirs('model', exist_ok=True)
    for f in os.listdir(os.path.join(drive_dir, 'model')):
        if f.endswith('.gguf') and not os.path.exists(os.path.join('model', f)):
            shutil.copy2(os.path.join(drive_dir, 'model', f), os.path.join('model', f))
            print(f'✓ Model hasil pelatihan termuat: {f}')

print('🚀 4. Menyalakan ulang Shiro...')
cuda_libs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib') + \
            glob.glob('/usr/lib/python*/dist-packages/nvidia/*/lib') + \
            ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']
valid_paths = [p for p in cuda_libs if os.path.isdir(p)]
if valid_paths:
    os.environ['LD_LIBRARY_PATH'] = ':'.join(valid_paths) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

from web import get_or_create_api_key
active_api_key = get_or_create_api_key()

os.environ['N_GPU_LAYERS'] = '-1'
os.environ['SHIRO_CONTEXT_SIZE'] = os.environ.get('SHIRO_CONTEXT_SIZE', '8192')

public_url = None
if TUNNEL_MODE == 'ngrok':
    from pyngrok import ngrok
    ngrok.kill()
    public_url = ngrok.connect(7474).public_url
else:
    if not os.path.exists('/content/cloudflared'):
        !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
        !chmod +x /content/cloudflared
    !rm -f /content/tunnel.log
    tunnel_proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7474'],
                                   stdout=open('/content/tunnel.log', 'w'),
                                   stderr=subprocess.STDOUT)

log_path = '/content/shiro_server.log'
with open(log_path, 'w', encoding='utf-8') as log_f:
    server_proc = subprocess.Popen(['python', 'web.py'], stdout=log_f, stderr=log_f)

if TUNNEL_MODE != 'ngrok':
    for _ in range(60):
        time.sleep(1)
        if os.path.exists('/content/tunnel.log'):
            with open('/content/tunnel.log', 'r') as f:
                content = f.read()
                match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
                if match:
                    public_url = match.group(1)
                    break

server_ready = False
for _ in range(120):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:7474/api/key', timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

clear_output(wait=True)

if server_ready and public_url:
    print('\n' + '='*62)
    print('🎉 SERVER SHIRO BERHASIL DI-RESTART (RERUN CEPAT)!')
    print('='*62)
    print(f'👉 BUKA WEB UI SHIRO DI SINI:')
    print(f'🔗 {public_url}')
    print('-'*62)
    print('✨ Seluruh sesi obrolan, ingatan, & dataset training aman 100%!')
    print(f'👉 SHIRO_BASE_URL = {public_url}/v1')
    print(f'👉 SHIRO_API_KEY  = {active_api_key}')
    print('='*62 + '\n')
    print('📁 Catatan: Log sistem tersimpan di shiro_server.log.\n')
    try:
        server_proc.wait()
    except KeyboardInterrupt:
        server_proc.terminate()
        if 'tunnel_proc' in locals():
            tunnel_proc.terminate()
        print('\nServer Shiro dihentikan.')
        if os.path.exists(drive_dir):
            for f in ['ingatan_shiro.json', 'isekai_world.json', 'api_key.json']:
                if os.path.exists(f):
                    shutil.copy2(f, os.path.join(drive_dir, f))
            if os.path.exists('training_data'):
                shutil.copytree('training_data', os.path.join(drive_dir, 'training_data'), dirs_exist_ok=True)
            print('✓ Seluruh sesi & data training Shiro tersinkron aman ke Google Drive.')
else:
    print(f'Public URL: {public_url}')
    print('❌ Gagal memulai server. Isi log terakhir:')
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            print(f.read()[-2000:])


## Langkah 7: 💾 Backup / Ekspor Manual ke Google Drive (Opsional)
Meskipun real-time auto-sync sudah aktif otomatis saat mengobrol, Kakak bisa menjalankan cell ini kapan saja untuk memastikan seluruh berkas (ingatan, profil, foto, dataset training, model adapter) tercadangkan rapi ke Google Drive.

In [ ]:
# Backup manual seluruh berkas ingatan, data training, profil & adapter ke Google Drive
import os
import shutil

DRIVE_DIR = '/content/drive/MyDrive/Shiro_Memory'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for f in ['ingatan_shiro.json', 'isekai_world.json', 'api_key.json']:
        if os.path.exists(f):
            shutil.copy2(f, os.path.join(DRIVE_DIR, f))
    if os.path.exists('profile'):
        shutil.copytree('profile', os.path.join(DRIVE_DIR, 'profile'), dirs_exist_ok=True)
    if os.path.exists('images'):
        shutil.copytree('images', os.path.join(DRIVE_DIR, 'images'), dirs_exist_ok=True)
    if os.path.exists('training_data'):
        shutil.copytree('training_data', os.path.join(DRIVE_DIR, 'training_data'), dirs_exist_ok=True)
    if os.path.exists('lora'):
        shutil.copytree('lora', os.path.join(DRIVE_DIR, 'lora'), dirs_exist_ok=True)
    if os.path.exists('model'):
        os.makedirs(os.path.join(DRIVE_DIR, 'model'), exist_ok=True)
        for f in os.listdir('model'):
            if 'shiroai-llm-v' in f.lower():
                shutil.copy2(os.path.join('model', f), os.path.join(DRIVE_DIR, 'model', f))
    print("=" * 62)
    print("🎉 SELURUH DATA, INGATAN & DATASET TRAINING BERHASIL DICADANGKAN!")
    print(f"📂 Folder Google Drive: {DRIVE_DIR}")
    print("=" * 62)
else:
    print("❌ Google Drive belum terhubung! Silakan jalankan Langkah 5 terlebih dahulu.")


## Langkah 8: 📦 Ekspor Dataset Pelatihan dari Percakapan Shiro
Jalankan cell ini untuk mengekstrak seluruh obrolan dan ingatan Shiro menjadi dataset training standar (ChatML, Alpaca, ShareGPT).
Ekspor riwayat ingatan Shiro ke dalam 2 folder terpisah:
- `training_data/chat/` -> Dataset Teks (ChatML, Alpaca, ShareGPT untuk Qwen3-32B, Qwen2.5-7B, dll.)
- `training_data/vision/` -> Dataset Multimodal (untuk Vision VLM Qwen2.5-VL-7B)

Dataset yang diekspor akan otomatis tersimpan langsung ke Google Drive (`/content/drive/MyDrive/Shiro_Memory/training_data/`) sehingga tidak akan hilang dan siap dipakai kapan saja untuk fine-tuning (Unsloth / LoRA)!

In [ ]:
# Ekspor ingatan dan percakapan Shiro ke format training
!python export_training_data.py

# Tampilkan cuplikan dataset training yang dihasilkan
import json
import os
import shutil

if os.path.exists("training_data/shiro_alpaca_train.json"):
    with open("training_data/shiro_alpaca_train.json", "r", encoding="utf-8") as f:
        sample = json.load(f)
    print(f"\n✓ Total data latih terkumpul: {len(sample)} pasang percakapan.")
    print("Contoh data pertama:", json.dumps(sample[0] if sample else {}, indent=2, ensure_ascii=False))

# Auto-backup dataset training ke Google Drive jika Drive terhubung
DRIVE_DIR = '/content/drive/MyDrive/Shiro_Memory'
if os.path.exists(DRIVE_DIR) and os.path.exists('training_data'):
    shutil.copytree('training_data', os.path.join(DRIVE_DIR, 'training_data'), dirs_exist_ok=True)
    print(f"✓ Dataset training otomatis dicadangkan ke Google Drive: {DRIVE_DIR}/training_data")


## Langkah 9: 🚀 Latih Model AI dengan Unsloth (Format ShiroAI-LLM-V1, V2, dst.) ⭐
Jalankan cell ini untuk melatih bobot model AI secara langsung menggunakan dataset percakapan yang diekspor di Langkah 8.

### 🏷️ Sistem Penomoran Versi Otomatis (Continuous Evolution):
- **Training Pertama** -> Otomatis menghasilkan Model AI: `ShiroAI-LLM-V1` (`model/ShiroAI-LLM-V1.gguf`).
- **Training Berikutnya** -> Jika `ShiroAI-LLM-V1` sudah ada, otomatis naik tingkat menjadi `ShiroAI-LLM-V2`, lalu `ShiroAI-LLM-V3`, dan seterusnya!
- **Langsung Terpajang di Web UI**: Model hasil training langsung muncul di menu **Pengaturan (⚙️) -> Model AI** sehingga Kakak bisa memilih versi mana saja yang ingin dipakai mengobrol.
- **Aman di Google Drive**: Seluruh file model `.gguf` dan adapter LoRA otomatis dicadangkan ke Google Drive (`Shiro_Memory/model/` dan `Shiro_Memory/lora/`).


In [ ]:
# ==============================================================================
# 🚀 1-KLIK TRAINING LORA SHIRO (HASIL MODEL: ShiroAI-LLM-V1, V2, dst.)
# ==============================================================================
import os, sys, shutil

# 1. Pastikan Google Drive terhubung untuk auto-backup model hasil training
print('🔄 1. Memeriksa koneksi Google Drive...')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Shiro_Memory'
os.makedirs(os.path.join(DRIVE_DIR, 'model'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'lora'), exist_ok=True)
os.environ['SHIRO_DRIVE_DIR'] = DRIVE_DIR

# 2. Pastikan dataset terbaru diekspor dari obrolan Shiro
%cd /content/ShiroLLM-AI-CHAT
print('\n📦 2. Memperbarui dataset training dari riwayat obrolan...')
!python export_training_data.py

# 3. Install Unsloth & dependensi fine-tuning (Hanya jika belum terinstall)
try:
    import unsloth
except ImportError:
    print('\n⚡ 3. Menginstall Unsloth & CUDA acceleration (Hanya 1x)...')
    !pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 4. Jalankan Pelatihan Model AI (Otomatis versi V1, V2, dst. + Auto Backup ke Drive)
print('\n🎓 4. Memulai pelatihan model AI...')
!python train_lora.py --steps 60 --lr 2e-4

# 5. Konfirmasi hasil backup model di Google Drive
print('\n' + '=' * 62)
print('🎉 STATUS MODEL HASIL TRAINING DI GOOGLE DRIVE:')
drive_model_dir = os.path.join(DRIVE_DIR, 'model')
if os.path.exists(drive_model_dir):
    found_models = [f for f in os.listdir(drive_model_dir) if f.endswith('.gguf')]
    if found_models:
        for m in sorted(found_models):
            sz = os.path.getsize(os.path.join(drive_model_dir, m)) / (1024**3)
            print(f'  ⭐ Model AI : {m} ({sz:.2f} GB)')
    else:
        print('  ℹ️ Bobot LoRA adapter tersimpan di Google Drive / Shiro_Memory / lora/')
print(f'📂 Lokasi Penyimpanan Drive: {DRIVE_DIR}')
print('=' * 62)
